# 实验6：Python 与 C 语言的前端映射

## 学习目标

1. 理解 PyAsc Python 前端为何精确映射 Ascend C API
2. 掌握 @overload / @require_jit / op_impl / OverloadDispatcher 分发机制
3. 从 Add 推导 Sub 的前端实现
4. 与编译原理"语法制导翻译"建立联系

## 环境准备

确认 PyAsc 源码已就绪：

In [ ]:
cd ~/pyasc
ls python/asc/language/basic/vec_binary.py
ls python/asc/language/basic/utils.py
ls python/test/unit/language/basic/test_vector_binary.py
!echo "源码就绪！" 

## 1. 阅读 Add 的三类重载

PyAsc 的 `add` 函数提供三种调用形式，对应 Ascend C 的 L0/L1/L2 API：

In [ ]:
cd ~/pyasc
!grep -A3 "overload" python/asc/language/basic/vec_binary.py | head -30

**三类重载对应：**
- **count 模式 (L2)**：`add(dst, src0, src1, count: int)` — 连续段计算
- **mask 连续模式 (L0)**：`add(dst, src0, src1, mask: int, repeat_times, repeat_params)`
- **mask 逐bit模式 (L1)**：`add(dst, src0, src1, mask: List[int], repeat_times, repeat_params)`

所有 overload 都带 `is_set_mask: bool = True` 参数。

> `@overload` 来自 `typing` 模块，只提供类型提示，不参与运行时逻辑。

## 2. Add 的实际实现

In [ ]:
# 查看 @require_jit 修饰的实现
!grep -A6 "^def add(dst" python/asc/language/basic/vec_binary.py

**核心调用链：**
```python
@require_jit                          # 只能在 JIT 上下文中调用
@set_binary_docstring(cpp_name="Add", append_text="按元素求和。")
def add(dst, src0, src1, *args, **kwargs):
    builder = global_builder.get_ir_builder()
    op_impl("add", dst, src0, src1, args, kwargs,
            builder.create_asc_AddL0Op,   # -> IR Operation
            builder.create_asc_AddL1Op,
            builder.create_asc_AddL2Op)
```

- `@require_jit`：确保只在 JIT 编译上下文中调用
- `@set_binary_docstring`：自动生成文档字符串（cpp_name="Add"）
- `global_builder.get_ir_builder()`：获取 MLIR Builder
- `builder.create_asc_AddL*Op`：创建 ASC-IR Operation（C++ 扩展暴露）

> Python 前端不生成文本代码，而是构建 IR 中间表示！

## 3. Sub ↔ Add 逐项对照

vec_binary.py 中包含 **19 种** 双目运算，每个都遵循相同的 3-overload + 1-implementation 模式：

In [ ]:
# 列出所有双目运算函数
!grep -n "set_binary_docstring" python/asc/language/basic/vec_binary.py

**Add vs Sub 对应表：**
- Python 函数名：`add` -> `sub`
- cpp_name：`"Add"` -> `"Sub"`
- append_text：`"按元素求和"` -> `"按元素求差"`
- L0/L1/L2 Builder：`AddL0Op/AddL1Op/AddL2Op` -> `SubL0Op/SubL1Op/SubL2Op`

> **思考：** 为什么 add/sub/mul/div/min/max 都共享完全相同的结构？

## 4. op_impl 分发机制

In [ ]:
# 查看 op_impl 实现
!grep -A40 "^def op_impl" python/asc/language/basic/utils.py | head -45

**分发逻辑（op_impl 内部）：**
- 创建 `OverloadDispatcher(callee_name)`
- L2：count 存在 -> 转 int32
- L0：mask 为 int -> 转 uint64, repeat_times 转 int8
- L1：mask 为 list -> 逐项转 uint64
- `dispatcher(*args, **kwargs)` 根据运行时参数类型选择分支

另外 `check_type()` 函数对 dst/src0/src1 做数据类型校验（如 add 支持 float16/float32/int16/int32）。

## 5. 运行前端单元测试

In [ ]:
cd ~/pyasc
!python3 -m pytest python/test/unit/language/basic/test_vector_binary.py -k "add_kernel or sub_kernel" -q -v

测试验证"前端能正确生成 IR 并进入编译流程"，不验证数值。

> **思考：** 为什么前端测试只验证流程？和实验5的 torch.allclose 如何分工？

## 6. 任务拓展

选 `mul` 或 `div`：
1. 找出 cpp_name 和 append_text
2. 记录 Builder 方法名
3. 定位对应 UT
4. 对比与 add 的异同

## 总结

1. Python 动态接口 vs Ascend C 静态 API 的 1:1 映射
2. @overload + op_impl + OverloadDispatcher = C++ 重载模拟
3. Add -> Sub 模板化推导
4. 前端映射 = 编译原理"语法制导翻译 + 中间代码生成"